# Training a perceptron with the perceptron loss

In [ ]:
import numpy as np
import plotly.graph_objects as go
from sklearn.datasets import make_classification
from sklearn.linear_model import Perceptron, SGDClassifier

# 100 points, two input columns, two well separated classes (labels 0 and 1)
X, y = make_classification(n_samples=100, n_features=2, n_informative=1, n_redundant=0, n_classes=2,
                           n_clusters_per_class=1, random_state=41, hypercube=False, class_sep=15)
print(X.shape, np.bincount(y))

## The four candidate losses on the worked example
Line 2x + 3y + 4 = 0; the point (4, 6) belongs to class -1 and (-2, -2) to class +1: both misclassified.

In [ ]:
w = np.array([2, 3]); b = 4
pts = np.array([[4, 6], [-2, -2]]); labels = np.array([-1, 1])
f = pts @ w + b
print("f(x):", f)
print("0-1 loss (count):", int((labels * f < 0).sum()))
print("sum of distances:", np.round(np.abs(f).sum() / np.linalg.norm(w), 2))
print("sum of |f(x)|:", np.abs(f).sum())
print("perceptron loss (average):", np.maximum(0, -labels * f).mean())

## The perceptron loss and its training loop

In [ ]:
y_pm = np.where(y == 1, 1, -1)      # labels 0/1 -> -1/+1

def loss(w1, w2, b):
    f = w1 * X[:, 0] + w2 * X[:, 1] + b
    return np.maximum(0, -y_pm * f).mean()

def perceptron(X, y, lr=0.1, epochs=1000, start=(-1.0, 1.0, 0.5)):
    w1, w2, b = start
    history = [loss(w1, w2, b)]
    for _ in range(epochs):
        for i in range(X.shape[0]):
            z = w1 * X[i, 0] + w2 * X[i, 1] + b
            # only a misclassified row changes the weights: w <- w + lr * y * x
            if y[i] * z < 0:
                w1 += lr * y[i] * X[i, 0]
                w2 += lr * y[i] * X[i, 1]
                b += lr * y[i]
                history.append(loss(w1, w2, b))
    return (w1, w2, b), history

(w1, w2, b), history = perceptron(X, y_pm)
print("w1, w2, b =", np.round([w1, w2, b], 3))
print("loss after each update:", np.round(history, 4))

In [ ]:
# the average loss after every update
go.Figure(go.Scatter(y=history, mode="lines+markers")).update_layout(
    template="simple_white", xaxis_title="update", yaxis_title="average perceptron loss")

## The line it found

In [ ]:
xs = np.linspace(-3.5, 2.5, 2)
fig = go.Figure()
for label, colour in [(1, "#54A24B"), (-1, "#4C78A8")]:
    m = y_pm == label
    fig.add_trace(go.Scatter(x=X[m, 0], y=X[m, 1], mode="markers", marker=dict(color=colour), name=f"y = {label}"))
fig.add_trace(go.Scatter(x=xs, y=-(w1 * xs + b) / w2, mode="lines", line=dict(color="#E45756", width=3), name="line"))
fig.update_layout(template="simple_white", yaxis_range=[-3.2, 2.4])

## Why the labels must be -1 and +1
With labels 0 and 1, `y[i] * z` is 0 for every class-0 row, so those rows never update the line.

In [ ]:
(w1b, w2b, bb), hist01 = perceptron(X, y)          # the 0/1 labels, unchanged
print("updates made:", len(hist01) - 1)
pred = (w1b * X[:, 0] + w2b * X[:, 1] + bb > 0).astype(int)
print("misclassified at the end:", (pred != y).sum(), "of", len(y))
# from the start (1, 1, 1) the same buggy loop happens to work
(w1c, w2c, bc), _ = perceptron(X, y, start=(1.0, 1.0, 1.0))
print("start (1, 1, 1), misclassified:", ((w1c * X[:, 0] + w2c * X[:, 1] + bc > 0).astype(int) != y).sum())

## scikit-learn: Perceptron and SGDClassifier(loss="perceptron")

In [ ]:
p = Perceptron(random_state=0).fit(X, y)
sgd = SGDClassifier(loss="perceptron", penalty=None, learning_rate="constant", eta0=0.1, random_state=0).fit(X, y)
print("Perceptron:   ", p.coef_[0].round(3), p.intercept_.round(3), "accuracy", p.score(X, y))
print("SGDClassifier:", sgd.coef_[0].round(3), sgd.intercept_.round(3), "accuracy", sgd.score(X, y))
# the same line: one set of weights is 10 times the other
print("ratio:", (p.coef_ / sgd.coef_).round(3), (p.intercept_ / sgd.intercept_).round(3))